# 03 — Business Analysis & Insights
Key questions with charts ready for Looker.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats

sns.set_theme(style='whitegrid')
df = pd.read_csv('../data/processed/gaming_clean.csv')
print(df.shape)

## Q1. Does gaming hurt grades?

In [ ]:
r, p = stats.pearsonr(df['gaming_hours'], df['grades'])
print(f'Pearson r = {r:.3f}, p = {p:.4f}')

fig = px.scatter(df.sample(1000, random_state=42),
    x='gaming_hours', y='grades', color='gaming_genre',
    trendline='ols', opacity=0.5,
    title=f'Gaming Hours vs Grades (r={r:.2f})')
fig.show()

## Q2. Addiction score vs academic performance

In [ ]:
order = ['Minimal', 'Moderate', 'High', 'Severe']
agg = df.groupby('addiction_level')['grades'].mean().reindex(order)
agg.plot(kind='bar', figsize=(8, 5), color='tomato', edgecolor='white')
plt.title('Avg Grades by Addiction Level')
plt.ylabel('Avg Grades')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Q3. Sleep quality impact

In [ ]:
order = ['Poor (<5h)', 'Insufficient (5-7h)', 'Adequate (7-9h)', 'Oversleeping (9h+)']
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
agg_sleep = df.groupby('sleep_quality').agg(avg_grades=('grades','mean'), avg_rt=('reaction_time_ms','mean')).reindex(order)
ax1.bar(order, agg_sleep['avg_grades'], color='steelblue', alpha=0.7)
ax2.plot(order, agg_sleep['avg_rt'], color='tomato', marker='o', linewidth=2)
ax1.set_ylabel('Avg Grades', color='steelblue')
ax2.set_ylabel('Avg Reaction Time (ms)', color='tomato')
plt.title('Sleep Quality vs Grades & Reaction Time')
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## Q4. Study hours vs grades — the strongest signal

In [ ]:
r2, p2 = stats.pearsonr(df['study_hours'], df['grades'])
print(f'Study hours ↔ grades: r = {r2:.3f}, p = {p2:.4f}')

fig = px.scatter(df.sample(1000, random_state=1),
    x='study_hours', y='grades', color='stress_level',
    trendline='ols', opacity=0.5,
    title=f'Study Hours vs Grades (r={r2:.2f})')
fig.show()

## Q5. Genre × stress heatmap

In [ ]:
pivot = df.pivot_table(values='grades', index='gaming_genre', columns='stress_level', aggfunc='mean')
pivot = pivot[['Low', 'Medium', 'High']]
plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', vmin=50, vmax=80)
plt.title('Avg Grades — Genre × Stress Level')
plt.tight_layout()
plt.show()